# Proyecto 02 - Inteligencia Artificial
*Rodrigo Sebastián Ajmac Aroche - 22279*

## Configuración 

**Librerías**

In [97]:
import time
import math
from random import randint

**Dependencias**

In [120]:
laberinto = r'C:\Users\ajmac\Documents\Universidad\Noveno semestre\IA\Proyecto 2\Proyecto_2_IA\Laberinto3-2.txt'

## Funciones Auxiliares

### Colas Implementadas en el Laboratorio 3 

In [121]:
class Nodo:
    def __init__(self, estado, padre=None, costo_camino=0, heuristica=0):
        self.estado = estado
        self.padre = padre
        self.costo_camino = costo_camino
        self.heuristica = heuristica

    def costo_total(self):
        # f(n) = g(n) + h(n) para el algoritmo A*
        return self.costo_camino + self.heuristica

    def __repr__(self):
        return f"[{self.estado}]"

In [122]:
class ColaFIFO:
    def __init__(self):
        self.lista = []

    def empty(self):
        return len(self.lista) == 0

    def top(self):
        return self.lista[0] if not self.empty() else None

    def pop(self):
        return self.lista.pop(0) if not self.empty() else None

    def add(self, elemento):
        # FIFO: Se añade al final, el primero en llegar (índice 0) es el primero en salir.
        self.lista.append(elemento)
        return self.lista

In [123]:
class ColaLIFO:
    def __init__(self):
        self.lista = []

    def empty(self):
        return len(self.lista) == 0

    def top(self):
        return self.lista[0] if not self.empty() else None

    def pop(self):
        return self.lista.pop(0) if not self.empty() else None

    def add(self, elemento):
        # LIFO: Se añade al inicio (índice 0), el último en llegar es el primero en salir.
        self.lista.insert(0, elemento)
        return self.lista

In [124]:
class ColaPrioridad:
    def __init__(self, tipo_prioridad='costo'):
        self.lista = []
        # El tipo determina por qué atributo ordenamos: 'costo', 'heuristica', o 'a_star'
        self.tipo_prioridad = tipo_prioridad

    def empty(self):
        return len(self.lista) == 0

    def top(self):
        return self.lista[0] if not self.empty() else None

    def pop(self):
        return self.lista.pop(0) if not self.empty() else None

    def add(self, nodo):
        self.lista.append(nodo)
        # Ordenamos de menor a mayor dependiendo de la métrica escogida
        if self.tipo_prioridad == 'costo':
            self.lista.sort(key=lambda x: x.costo_camino)
        elif self.tipo_prioridad == 'heuristica':
            self.lista.sort(key=lambda x: x.heuristica)
        elif self.tipo_prioridad == 'a_star':
            self.lista.sort(key=lambda x: x.costo_total())
        return self.lista

### Funciones para Abrir .txt, Calcular Branching Factor, Generar Puntos de Inicio y Salida Aleatorios y Obtener Vecinos

In [125]:
def leer_laberinto(ruta_archivo):
    """
    Lee un archivo de texto y lo convierte en una matriz bidimensional.
    Retorna la matriz del laberinto, una lista con los puntos de partida y 
    una lista con los puntos de salida.
    """
    laberinto = []
    puntos_partida = []
    puntos_salida = []

    with open(ruta_archivo, 'r') as archivo:
        for i, linea in enumerate(archivo):
            # Limpiamos los saltos de línea y espacios
            fila_texto = linea.strip()
            
            # Si la línea está vacía, la saltamos
            if not fila_texto:
                continue
                
            fila_int = []
            for j, caracter in enumerate(fila_texto):
                valor = int(caracter)
                fila_int.append(valor)
                
                # Identificar puntos clave
                if valor == 2:
                    puntos_partida.append((i, j))
                elif valor == 3:
                    puntos_salida.append((i, j))
                    
            laberinto.append(fila_int)

    return laberinto, puntos_partida, puntos_salida

Supongamos que $b*$ es el número promedio de caminos disponibles por nodo. Notemos que $b^*\leq 4$ ya que cada nodo a lo más puede tener 4 vecinos por lo que si $b^*>4$ entonces uno o más nodos tendrían más de $4$ vecinos lo cual es imposible. Ahora bien, denotemos $N$ como el número total de nodos explorados y $d$ como el largo del camino o la profundidad. 

Notemos que podemos expresar $N$ como,
$$
N = b^* + (b^*)^2 + ... + (b^*)^d
$$

ya que por cada nivel $i$ vamos a tener $(b^*)^i$ nodos disponibles que provienen de los $(b^*)^{i-1}$ nodos del nivel anterior con los $b^*$ caminos disponibles por nodos. 

La expresión de arriba se puede reescribir como,
$$
N = \frac{(b^*)^{d-1} - b^*}{b^* - 1}
$$

Con esta expresión es más fácil aproximar $b^*$ utilizando el concepto de bisección.

In [126]:
def calcular_branching_factor(nodos_explorados, profundidad, tolerancia=0.0001):
    """
    Aproxima el factor de ramificación efectivo (b*) usando búsqueda binaria.
    Maneja desbordamientos de memoria para caminos muy largos.
    """
    if profundidad == 0:
        return 0.0
    if profundidad == 1:
        return float(nodos_explorados)

    # Rango inicial: en un laberinto de 4 movimientos, b* nunca excederá 4.0
    low = 1.0
    high = 4.0 

    while high - low > tolerancia:
        mid = (low + high) / 2.0
        
        try:
            # Intentamos calcular los nodos generados teóricos
            if mid == 1.0:
                calculado = profundidad
            else:
                calculado = (mid ** (profundidad + 1) - mid) / (mid - 1)

            # Si el cálculo es exitoso, ajustamos los límites normalmente
            if calculado < nodos_explorados:
                low = mid
            else:
                high = mid
                
        except OverflowError:
            # Si el número es tan grande que desborda la memoria de Python (Overflow),
            # es definitivamente mayor que 'nodos_explorados'. 
            # Por lo tanto, el valor 'mid' que probamos era demasiado alto.
            high = mid

    # Retornar el punto medio como la mejor aproximación
    return (low + high) / 2.0

In [127]:
def gen_start_n_final_random(lista_1: list, lista_2:list):
    x_1 = len(lista_1) - 1
    x_2  = len(lista_2) - 1

    ind_1 = randint(0,x_1)
    ind_2 = randint(0,x_2)

    return ind_1, ind_2

#### Función para Obtener Vecinos

In [128]:
def obtener_vecinos(estado, laberinto):
    """
    Retorna los vecinos válidos de una coordenada (fila, columna)
    respetando la jerarquía: Arriba, Derecha, Abajo, Izquierda.
    """
    fila, col = estado
    vecinos_validos = []
    
    # Jerarquía de movimientos: (delta_fila, delta_columna)
    # Arriba: (-1, 0), Derecha: (0, 1), Abajo: (1, 0), Izquierda: (0, -1)
    movimientos = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    
    filas_totales = len(laberinto)
    cols_totales = len(laberinto[0])
    
    for df, dc in movimientos:
        n_fila = fila + df
        n_col = col + dc
        
        # Validar que el vecino esté dentro de los límites del laberinto
        if 0 <= n_fila < filas_totales and 0 <= n_col < cols_totales:
            # Validar que el vecino no sea una pared (1)
            if laberinto[n_fila][n_col] != 1:
                vecinos_validos.append((n_fila, n_col))
                
    return vecinos_validos

## Implementación de Algoritmos

### Búsqueda No Informada

#### Breadth First Search (BFS)

In [129]:
def bfs(laberinto, inicio, salida):
    """
    Algoritmo de búsqueda en anchura (Breadth First Search).
    Encuentra el camino más corto en grafos no ponderados.
    """
    tiempo_inicio = time.time()
    
    # Inicializamos el nodo raíz
    nodo_inicial = Nodo(estado=inicio, costo_camino=0)
    
    # Usamos la clase ColaFIFO que creaste
    frontera = ColaFIFO()
    frontera.add(nodo_inicial)
    
    # Usamos un set para búsquedas O(1) de los estados explorados
    explorados = set()
    explorados.add(inicio)
    
    nodos_explorados_count = 0
    
    while not frontera.empty():
        # Sacamos el primer elemento que entró a la cola
        nodo_actual = frontera.pop()
        nodos_explorados_count += 1
        
        # 1. Comprobar si es la meta
        if nodo_actual.estado == salida:
            tiempo_fin = time.time()
            
            # Reconstruir el camino desde la meta hasta el inicio
            camino = []
            nodo_temp = nodo_actual
            while nodo_temp is not None:
                camino.append(nodo_temp.estado)
                nodo_temp = nodo_temp.padre
            
            # Invertimos la lista para que el camino vaya desde inicio a fin
            camino.reverse() 
            
            return {
                'camino': camino,
                'longitud': len(camino) - 1, # Pasos dados (sin contar la casilla inicial)
                'nodos_explorados': nodos_explorados_count,
                'tiempo': tiempo_fin - tiempo_inicio
            }
            
        # 2. Expandir los vecinos
        vecinos = obtener_vecinos(nodo_actual.estado, laberinto)
        
        for estado_vecino in vecinos:
            if estado_vecino not in explorados:
                # Marcamos como explorado al añadir a la frontera para evitar duplicados
                explorados.add(estado_vecino) 
                
                # Creamos el nodo hijo, apuntando al actual como su padre
                hijo = Nodo(
                    estado=estado_vecino, 
                    padre=nodo_actual, 
                    costo_camino=nodo_actual.costo_camino + 1
                )
                frontera.add(hijo)
                
    # Si la cola se vacía y nunca retornamos, significa que no hay solución
    tiempo_fin = time.time()
    return {
        'camino': None,
        'longitud': 0,
        'nodos_explorados': nodos_explorados_count,
        'tiempo': tiempo_fin - tiempo_inicio
    }

#### Depth First Search (DFS)

In [130]:
def dfs(laberinto, inicio, salida):
    """
    Algoritmo de búsqueda en profundidad (Depth First Search).
    Explora un camino hasta el final antes de retroceder.
    """
    tiempo_inicio = time.time()
    
    # Inicializamos el nodo raíz
    nodo_inicial = Nodo(estado=inicio, costo_camino=0)
    
    # ¡Aquí está la magia! Usamos la clase ColaLIFO que creaste
    frontera = ColaLIFO()
    frontera.add(nodo_inicial)
    
    # Set para búsquedas rápidas de los estados explorados
    explorados = set()
    
    nodos_explorados_count = 0
    
    while not frontera.empty():
        # Sacamos el último elemento que entró a la cola
        nodo_actual = frontera.pop()
        
        # Si el estado ya fue explorado, lo saltamos
        if nodo_actual.estado in explorados:
            continue
            
        # Lo marcamos como explorado al sacarlo
        explorados.add(nodo_actual.estado)
        nodos_explorados_count += 1
        
        # 1. Comprobar si es la meta
        if nodo_actual.estado == salida:
            tiempo_fin = time.time()
            
            # Reconstruir el camino desde la meta hasta el inicio
            camino = []
            nodo_temp = nodo_actual
            while nodo_temp is not None:
                camino.append(nodo_temp.estado)
                nodo_temp = nodo_temp.padre
            
            camino.reverse() # Invertimos para que vaya de inicio a fin
            
            return {
                'camino': camino,
                'longitud': len(camino) - 1,
                'nodos_explorados': nodos_explorados_count,
                'tiempo': tiempo_fin - tiempo_inicio
            }
            
        # 2. Expandir los vecinos
        vecinos = obtener_vecinos(nodo_actual.estado, laberinto)
        
        # INVERTIMOS LOS VECINOS: Para que al insertarlos en la ColaLIFO, 
        # "Arriba" quede en el tope y sea el primero en ser evaluado.
        vecinos.reverse()
        
        for estado_vecino in vecinos:
            if estado_vecino not in explorados:
                hijo = Nodo(
                    estado=estado_vecino, 
                    padre=nodo_actual, 
                    costo_camino=nodo_actual.costo_camino + 1
                )
                frontera.add(hijo)
                
    # Si no se encuentra solución
    tiempo_fin = time.time()
    return {
        'camino': None,
        'longitud': 0,
        'nodos_explorados': nodos_explorados_count,
        'tiempo': tiempo_fin - tiempo_inicio
    }

### Búsqueda Informada

#### Funciones Heurísticas

In [131]:
def distancia_manhattan(estado_actual, estado_meta):
    """
    Calcula la distancia de Manhattan entre dos puntos (filas, columnas).
    Fórmula: |x1 - x2| + |y1 - y2|
    """
    fila_actual, col_actual = estado_actual
    fila_meta, col_meta = estado_meta
    return abs(fila_actual - fila_meta) + abs(col_actual - col_meta)

def distancia_euclidiana(estado_actual, estado_meta):
    """
    Calcula la distancia Euclidiana (línea recta) entre dos puntos.
    Fórmula: sqrt((x1 - x2)^2 + (y1 - y2)^2)
    """
    fila_actual, col_actual = estado_actual
    fila_meta, col_meta = estado_meta
    return math.sqrt((fila_actual - fila_meta)**2 + (col_actual - col_meta)**2)

#### Greedy First Search

In [132]:
def greedy_first_search(laberinto, inicio, salida, heuristica_func):
    """
    Algoritmo Greedy First Search.
    Utiliza una función heurística para priorizar los nodos que parecen
    estar más cerca de la meta.
    """
    tiempo_inicio = time.time()
    
    # Calculamos la heurística del nodo inicial
    h_inicial = heuristica_func(inicio, salida)
    nodo_inicial = Nodo(estado=inicio, costo_camino=0, heuristica=h_inicial)
    
    # Usamos tu ColaPrioridad ordenada por 'heuristica'
    frontera = ColaPrioridad(tipo_prioridad='heuristica')
    frontera.add(nodo_inicial)
    
    # Set para estados explorados
    explorados = set()
    nodos_explorados_count = 0
    
    while not frontera.empty():
        # Sacamos el nodo con la heurística más baja (el que parece más cerca)
        nodo_actual = frontera.pop()
        
        # Si ya lo exploramos con una mejor prioridad antes, lo saltamos
        if nodo_actual.estado in explorados:
            continue
            
        explorados.add(nodo_actual.estado)
        nodos_explorados_count += 1
        
        # 1. Comprobar si es la meta
        if nodo_actual.estado == salida:
            tiempo_fin = time.time()
            
            camino = []
            nodo_temp = nodo_actual
            while nodo_temp is not None:
                camino.append(nodo_temp.estado)
                nodo_temp = nodo_temp.padre
            
            camino.reverse() 
            
            return {
                'camino': camino,
                'longitud': len(camino) - 1,
                'nodos_explorados': nodos_explorados_count,
                'tiempo': tiempo_fin - tiempo_inicio
            }
            
        # 2. Expandir los vecinos
        vecinos = obtener_vecinos(nodo_actual.estado, laberinto)
        
        for estado_vecino in vecinos:
            if estado_vecino not in explorados:
                # Calculamos la heurística para el nuevo vecino
                h_vecino = heuristica_func(estado_vecino, salida)
                
                hijo = Nodo(
                    estado=estado_vecino, 
                    padre=nodo_actual, 
                    costo_camino=nodo_actual.costo_camino + 1,
                    heuristica=h_vecino
                )
                # Al añadirlo, la cola lo ordenará automáticamente por su atributo 'heuristica'
                frontera.add(hijo)
                
    # Si no hay solución
    tiempo_fin = time.time()
    return {
        'camino': None,
        'longitud': 0,
        'nodos_explorados': nodos_explorados_count,
        'tiempo': tiempo_fin - tiempo_inicio
    }

#### A*

In [133]:
def a_star_search(laberinto, inicio, salida, heuristica_func):
    """
    Algoritmo A* (A-Star).
    Encuentra el camino más corto evaluando f(n) = g(n) + h(n).
    """
    # Usamos perf_counter para una medición de tiempo más precisa
    tiempo_inicio = time.perf_counter()
    
    # 1. Configurar nodo inicial
    h_inicial = heuristica_func(inicio, salida)
    nodo_inicial = Nodo(estado=inicio, costo_camino=0, heuristica=h_inicial)
    
    # Usamos ColaPrioridad ordenada por 'a_star' (f(n))
    frontera = ColaPrioridad(tipo_prioridad='a_star')
    frontera.add(nodo_inicial)
    
    # Diccionario para guardar el costo g(n) más bajo encontrado para cada estado
    costos_g = {inicio: 0}
    
    nodos_explorados_count = 0
    
    while not frontera.empty():
        # Sacamos el nodo con el menor f(n)
        nodo_actual = frontera.pop()
        
        # Si sacamos un nodo pero ya habíamos encontrado un camino más corto hacia él, lo ignoramos
        if nodo_actual.costo_camino > costos_g.get(nodo_actual.estado, float('inf')):
            continue
            
        nodos_explorados_count += 1
        
        # 2. Comprobar si es la meta
        if nodo_actual.estado == salida:
            tiempo_fin = time.perf_counter()
            
            camino = []
            nodo_temp = nodo_actual
            while nodo_temp is not None:
                camino.append(nodo_temp.estado)
                nodo_temp = nodo_temp.padre
            
            camino.reverse() 
            
            return {
                'camino': camino,
                'longitud': len(camino) - 1,
                'nodos_explorados': nodos_explorados_count,
                'tiempo': tiempo_fin - tiempo_inicio
            }
            
        # 3. Expandir los vecinos
        vecinos = obtener_vecinos(nodo_actual.estado, laberinto)
        
        for estado_vecino in vecinos:
            # El costo g(n) para el vecino es el costo actual + 1 paso
            nuevo_costo_g = nodo_actual.costo_camino + 1
            
            # Si es la primera vez que vemos al vecino, o si encontramos un camino más rápido hacia él
            if estado_vecino not in costos_g or nuevo_costo_g < costos_g[estado_vecino]:
                
                # Actualizamos el mejor costo conocido para este vecino
                costos_g[estado_vecino] = nuevo_costo_g
                
                # Calculamos su heurística h(n)
                h_vecino = heuristica_func(estado_vecino, salida)
                
                hijo = Nodo(
                    estado=estado_vecino, 
                    padre=nodo_actual, 
                    costo_camino=nuevo_costo_g,
                    heuristica=h_vecino
                )
                
                frontera.add(hijo)
                
    # Si la cola se vacía y no hay solución
    tiempo_fin = time.perf_counter()
    return {
        'camino': None,
        'longitud': 0,
        'nodos_explorados': nodos_explorados_count,
        'tiempo': tiempo_fin - tiempo_inicio
    }

## Resultados

### Configuración del Laberinto

**Leer Laberinto**

In [134]:
laberinto, inicios, salidas = leer_laberinto(laberinto)

**Definir Punto de Inicio y Salida**

In [135]:
ind_1, ind_2 = gen_start_n_final_random(inicios, salidas)

In [136]:
print(f"Inicio Aleatorio: {inicios[ind_1]} \n Salida Aleatoria: {salidas[ind_2]}")

Inicio Aleatorio: (28, 57) 
 Salida Aleatoria: (126, 62)


In [137]:
punto_inicio = inicios[0]
punto_salida = salidas[0]

### Resultado BFS

In [138]:
resultados_bfs = bfs(laberinto, punto_inicio, punto_salida)
    
# Calcular branching factor
b_star = calcular_branching_factor(
    resultados_bfs['nodos_explorados'], 
    resultados_bfs['longitud']
)
    
# Mostrar resultados
print("--- Resultados BFS ---")
print(f"Longitud del camino: {resultados_bfs['longitud']}")
print(f"Nodos explorados: {resultados_bfs['nodos_explorados']}")
print(f"Tiempo de ejecución: {resultados_bfs['tiempo']:.10f} segundos")
print(f"Branching factor (b*): {b_star:.4f}")

--- Resultados BFS ---
Longitud del camino: 183
Nodos explorados: 3805
Tiempo de ejecución: 0.0155887604 segundos
Branching factor (b*): 1.0251


### Resultado DFS

In [139]:
resultados_dfs = dfs(laberinto, punto_inicio, punto_salida)

if resultados_dfs['camino']:
    b_star_dfs = calcular_branching_factor(
        resultados_dfs['nodos_explorados'], 
        resultados_dfs['longitud']
    )
    
    print("\n--- Resultados DFS ---")
    print(f"Longitud del camino: {resultados_dfs['longitud']}")
    print(f"Nodos explorados: {resultados_dfs['nodos_explorados']}")
    print(f"Tiempo de ejecución: {resultados_dfs['tiempo']:.10f} segundos")
    print(f"Branching factor (b*): {b_star_dfs:.4f}")


--- Resultados DFS ---
Longitud del camino: 891
Nodos explorados: 1079
Tiempo de ejecución: 0.3465793133 segundos
Branching factor (b*): 1.0004


### Resultados Greedy First Search

In [140]:
resultados_greedy_manhattan = greedy_first_search(
    laberinto, punto_inicio, punto_salida, distancia_manhattan
)

if resultados_greedy_manhattan['camino']:
    b_star_g_manhattan = calcular_branching_factor(
        resultados_greedy_manhattan['nodos_explorados'], 
        resultados_greedy_manhattan['longitud']
    )
    print("\n--- Resultados Greedy (Manhattan) ---")
    print(f"Longitud del camino: {resultados_greedy_manhattan['longitud']}")
    print(f"Nodos explorados: {resultados_greedy_manhattan['nodos_explorados']}")
    print(f"Tiempo: {resultados_greedy_manhattan['tiempo']:.10f} s")
    print(f"Branching factor (b*): {b_star_g_manhattan:.4f}")

# Prueba con Distancia Euclidiana
resultados_greedy_euclidiana = greedy_first_search(
    laberinto, punto_inicio, punto_salida, distancia_euclidiana
)

if resultados_greedy_euclidiana['camino']:
    b_star_g_euclidiana = calcular_branching_factor(
        resultados_greedy_euclidiana['nodos_explorados'], 
        resultados_greedy_euclidiana['longitud']
    )
    print("\n--- Resultados Greedy (Euclidiana) ---")
    print(f"Longitud del camino: {resultados_greedy_euclidiana['longitud']}")
    print(f"Nodos explorados: {resultados_greedy_euclidiana['nodos_explorados']}")
    print(f"Tiempo: {resultados_greedy_euclidiana['tiempo']:.10f} s")
    print(f"Branching factor (b*): {b_star_g_euclidiana:.4f}")


--- Resultados Greedy (Manhattan) ---
Longitud del camino: 187
Nodos explorados: 1335
Tiempo: 0.0130484104 s
Branching factor (b*): 1.0170

--- Resultados Greedy (Euclidiana) ---
Longitud del camino: 187
Nodos explorados: 607
Tiempo: 0.0084750652 s
Branching factor (b*): 1.0108


### Resultados A*

In [141]:
resultados_astar_manhattan = a_star_search(
    laberinto, punto_inicio, punto_salida, distancia_manhattan
)

if resultados_astar_manhattan['camino']:
    b_star_astar_m = calcular_branching_factor(
        resultados_astar_manhattan['nodos_explorados'], 
        resultados_astar_manhattan['longitud']
    )
    print("\n--- Resultados A* (Manhattan) ---")
    print(f"Longitud del camino: {resultados_astar_manhattan['longitud']}")
    print(f"Nodos explorados: {resultados_astar_manhattan['nodos_explorados']}")
    print(f"Tiempo: {resultados_astar_manhattan['tiempo']:.10f} s")
    print(f"Branching factor (b*): {b_star_astar_m:.4f}")

# Prueba A* con Distancia Euclidiana
resultados_astar_euclidiana = a_star_search(
    laberinto, punto_inicio, punto_salida, distancia_euclidiana
)

if resultados_astar_euclidiana['camino']:
    b_star_astar_e = calcular_branching_factor(
        resultados_astar_euclidiana['nodos_explorados'], 
        resultados_astar_euclidiana['longitud']
    )
    print("\n--- Resultados A* (Euclidiana) ---")
    print(f"Longitud del camino: {resultados_astar_euclidiana['longitud']}")
    print(f"Nodos explorados: {resultados_astar_euclidiana['nodos_explorados']}")
    print(f"Tiempo: {resultados_astar_euclidiana['tiempo']:.10f} s")
    print(f"Branching factor (b*): {b_star_astar_e:.4f}")


--- Resultados A* (Manhattan) ---
Longitud del camino: 183
Nodos explorados: 2050
Tiempo: 0.0262162000 s
Branching factor (b*): 1.0207

--- Resultados A* (Euclidiana) ---
Longitud del camino: 183
Nodos explorados: 2479
Tiempo: 0.0316931000 s
Branching factor (b*): 1.0221
